# Load and initialise

In [ ]:
import sys
sys.path.append('..')
import torch
from tqdm import tqdm
from models import GenTeacher, GenStudent, GenClassifier, vit_parameters, classifier_layels_list
from models.model import AttentionAutoEncoder
from utils import save_data
from utils.dataset import get_datasets, image_transform
from utils.channels import Channels

device = 'cuda:0'
model = GenTeacher('ViT-B/32', device)
classifier = GenClassifier(512, 100, classifier_layels_list, device)
atten_ae = AttentionAutoEncoder(512, 2).to(device)
model.load_state_dict(torch.load('../results/weights/retrain_teacher/cifar100_teacher_model_sota.pt'))
classifier.load_state_dict(torch.load('../results/weights/retrain_teacher/cifar100_classifier_teacher_model_sota.pt'))
# atten_ae.load_state_dict(torch.load('../results/weights/joint_training_with_ae/atten_ae_239.pt'))
_, test_loader = get_datasets(image_transform, 1000, 'cifar100')

# Test at different SNRs

In [ ]:
model.eval()
classifier.eval()
channels = Channels(device)

ACC_with_SNR = []
for snr in range(-10, 25, 2):
    test_loader = tqdm(test_loader, desc='Testing on SNR:'+str(snr), ncols=120)
    Length = 0
    acc = 0
    for i, (images, labels) in enumerate(test_loader):
        images, labels = images.to(device), labels.to(device)

        with torch.no_grad():

            features = model(images)
            # features = atten_ae.encoder(features)
            features_awgn = channels.AWGN(features, snr)
            # features_awgn = atten_ae.decoder(features_awgn, features_awgn)
            logits = classifier(features_awgn)

            preds = logits.argmax(dim=1)
            acc += (preds == labels).float().sum()

        Length = (i+1)*len(labels)

        test_loader.set_postfix(Acc = acc.item() / Length)
        test_loader.update()
    ACC_with_SNR.append(acc.item() / Length)

print(ACC_with_SNR)

# Save data

In [ ]:
save_data(ACC_with_SNR, '../results/curve_data/ACC_with_SNR/ACC_with_SNR.txt')